# `vknumpy` Demo — NumPy-Compatible Arrays on Vulkan GPU

Showcases GPU array creation, elementwise ufuncs, matrix multiplication (`@` / `einsum`), linear algebra, and zero-copy Vulkan SPIR-V `bincount` / `histogram` / `histogram2d`.

In [1]:
import sys, time
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vknumpy").is_dir()), Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import vknumpy as vnp

print("Vulkan device:", vnp.device_info())

# 1024x1024 float32 GEMM: CPU NumPy vs Vulkan GPU vknumpy
rng = np.random.default_rng(42)
a_np = rng.standard_normal((1024, 1024), dtype=np.float32)
b_np = rng.standard_normal((1024, 1024), dtype=np.float32)
a_vk, b_vk = vnp.array(a_np), vnp.array(b_np)
_ = (a_vk @ b_vk).to_numpy()  # warmup

t0 = time.perf_counter()
c_np = a_np @ b_np
cpu_ms = (time.perf_counter() - t0) * 1000.0

t0 = time.perf_counter()
c_vk = (a_vk @ b_vk).to_numpy()
gpu_ms = (time.perf_counter() - t0) * 1000.0

max_err = float(np.max(np.abs(c_np - c_vk)))
print(f"1024x1024 GEMM | CPU: {cpu_ms:.1f} ms | Vulkan GPU: {gpu_ms:.1f} ms | max_abs_err={max_err:.2e}")

Vulkan device: {'backend': 'ggml-vulkan', 'vulkan': True, 'name': 'AMD Custom GPU 0405 (RADV VANGOGH)', 'free_memory_mb': 9635.41, 'total_memory_mb': 12288.0}
1024x1024 GEMM | CPU: 16.7 ms | Vulkan GPU: 26.6 ms | max_abs_err=1.98e-04


ggml_vulkan: Found 1 Vulkan devices:
ggml_vulkan: 0 = AMD Custom GPU 0405 (RADV VANGOGH) (radv) | uma: 1 | rebar: 1 | fp16: 0 | bf16: 0 | fp4: 0 | warp size: 32 | shared memory: 65536 | int dot: 1 | matrix cores: none


In [2]:
# Elementwise ufuncs, reductions, einsum, and linear algebra on GPU
x = vnp.linspace(-3.0, 3.0, 1_000_000, dtype=np.float32)
y = vnp.sin(x) * vnp.exp(-0.2 * x * x)
print(f"1M ufunc | mean={float(vnp.mean(y).item()):.6f} | std={float(vnp.std(y).item()):.6f} | max={float(vnp.max(y).item()):.6f}")

# Batched einsum contraction
M = vnp.array(rng.standard_normal((8, 64, 32), dtype=np.float32))
N = vnp.array(rng.standard_normal((8, 32, 16), dtype=np.float32))
P = vnp.einsum("bij,bjk->bik", M, N)
print("Batched einsum shape:", P.shape)

# Linear solve Ax = b
A = vnp.array(np.eye(64, dtype=np.float32) * 5.0 + 0.1 * rng.standard_normal((64, 64)).astype(np.float32))
b = vnp.array(rng.standard_normal((64,), dtype=np.float32))
sol = vnp.linalg.solve(A, b)
resid = float(vnp.linalg.norm(A @ sol - b).item())
print(f"64x64 linear solve residual: {resid:.2e}")

[0 AMD Custom GPU 0405 (RADV VANGOGH)]  queueC=1[4]  queueT=0[1]  prefer-slm=0  rebar=1  r-score=11
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  fp16-p/s/u/a=1/1/1/1  int8-p/s/u/a=1/1/1/1  bf16-p/s=1/0
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  subgroup=64(32~64)  ops=1/1/1/1/1/1/1/1/1/1
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  fp16-cm=0  int8-cm=0  bf16-cm=0  fp8-cm=0  fp6-cm=0  fp4-cm=0


1M ufunc | mean=-0.000000 | std=0.462609 | max=0.700662
Batched einsum shape: (8, 64, 16)
64x64 linear solve residual: 5.95e-07


In [3]:
# Zero-copy Vulkan SPIR-V bincount, 1D histogram, and 2D histogram on 5M samples
n = 5_000_000
gx = vnp.array(rng.normal(0.0, 1.0, size=n).astype(np.float32))
gy = vnp.array(rng.normal(0.0, 1.0, size=n).astype(np.float32))
cats = vnp.array(rng.integers(0, 16, size=n, dtype=np.int32))

t0 = time.perf_counter()
bc = vnp.bincount(cats, minlength=16)
bc_ms = (time.perf_counter() - t0) * 1000.0

t0 = time.perf_counter()
h1, edges = vnp.histogram(gx, bins=50, range=(-4.0, 4.0))
h1_ms = (time.perf_counter() - t0) * 1000.0

t0 = time.perf_counter()
h2, _, _ = vnp.histogram2d(gx, gy, bins=(256, 256), range=((-3.5, 3.5), (-3.5, 3.5)))
h2_ms = (time.perf_counter() - t0) * 1000.0

print(f"5M GPU bincount:    {bc_ms:.2f} ms (sum={int(bc.to_numpy().sum()):,})")
print(f"5M GPU histogram:   {h1_ms:.2f} ms (in-range={int(h1.to_numpy().sum()):,})")
print(f"5M GPU histogram2d: {h2_ms:.2f} ms (peak bin={int(h2.to_numpy().max()):,})")

5M GPU bincount:    45.56 ms (sum=5,000,000)
5M GPU histogram:   3.17 ms (in-range=4,999,674)
5M GPU histogram2d: 6.74 ms (peak bin=653)
